# 06 - SFT dataset generation


## Goal

Run the rule-based generator over the graph, inspect three example records, and learn the JSONL schema every fine-tune in this lab consumes.


## Prerequisites

- Notebooks 01-04 read; you should know what each entity type means.
- The generator lives at `src/dataset/qa_generator.py`. Each template is small enough to read in full.


## Environment bootstrap

This cell makes the notebook portable between a local checkout and Colab.

- **Local**: when the notebook lives inside the repo, we add the repo root to `sys.path`
  so the `src` package imports cleanly.
- **Colab**: the import will fail with `ModuleNotFoundError`. We catch that and print a
  one-line reminder showing the `git clone` the learner should run. We deliberately do
  **not** execute the clone for them — the lab policy is *recipes only, no auto-downloads*.


In [ ]:
import sys
from pathlib import Path

try:
    # Local checkout: walk up from the notebook to the repo root.
    here = Path.cwd()
    for candidate in [here, *here.parents]:
        if (candidate / "src" / "common" / "paths.py").exists():
            if str(candidate) not in sys.path:
                sys.path.insert(0, str(candidate))
            break
    from src.common.paths import REPO_ROOT, MINI_REPO, ensure_dirs
    ensure_dirs()
    print(f"repo root: {REPO_ROOT}")
    print(f"mini repo: {MINI_REPO}")
except ModuleNotFoundError:
    print("`src` not importable. If you are on Colab, run this in a separate cell:")
    print("    !git clone https://example.invalid/kde_ontology_slm_lab.git")
    print("    %cd kde_ontology_slm_lab")
    print("Then re-run this cell. We will not auto-clone for you (lab policy:")
    print("recipes only, no auto-downloads).")


## 1. Rebuild the graph


In [ ]:
from src.common.paths import MINI_REPO, DATASETS_OUT_DIR
from src.repo_ingest.scanner import scan
from src.repo_ingest.cmake_reader import read_cmake
from src.repo_ingest.cpp_reader import read_cpp
from src.repo_ingest.qml_reader import read_qml
from src.repo_ingest.dbus_reader import read_dbus
from src.repo_ingest.kconfig_reader import read_kconfig
from src.repo_ingest.desktop_file_reader import read_desktop
from src.repo_ingest.log_reader import read_log
from src.ontology.extractor import (
    ExtractionBundle, from_cmake, from_cpp, from_qml, from_dbus,
    from_kconfig, from_desktop, from_log,
)
from src.ontology.schema import Entity
from src.common.ids import make_id
from src.graph.builder import build_graph

rep = scan(MINI_REPO)
b = ExtractionBundle()
rid = b.add_entity(Entity(id=make_id('Repository', MINI_REPO.name),
                          type='Repository', name=MINI_REPO.name,
                          source_path=str(MINI_REPO)))
for sf in rep.by_kind('cmake'): from_cmake(b, read_cmake(sf.path), rid)
for sf in rep.by_kind('cpp_header') + rep.by_kind('cpp_source'): from_cpp(b, read_cpp(sf.path))
for sf in rep.by_kind('qml'): from_qml(b, read_qml(sf.path))
for sf in rep.by_kind('dbus'): from_dbus(b, read_dbus(sf.path))
for sf in rep.by_kind('kconfig'): from_kconfig(b, read_kconfig(sf.path))
for sf in rep.by_kind('desktop'): from_desktop(b, read_desktop(sf.path))
for sf in rep.by_kind('log'): from_log(b, read_log(sf.path))
g = build_graph(b)


## 2. Generate examples

`generate(g)` walks every template (signal-emission, config-keys, qml-backend, log-to-component, dbus-methods, refusal) and yields deduplicated `dict` records.


In [ ]:
from src.dataset.qa_generator import generate, TEMPLATES

records = generate(g)
print(f'{len(records)} records produced by {len(TEMPLATES)} templates')
from collections import Counter
print('by task_type:')
for k, n in Counter(r['task_type'] for r in records).most_common():
    print(f'  {n:3d}  {k}')


## 3. Inspect three example records

We pick one from three different task types so you can compare instruction styles and how `evidence` always points back at the graph.


In [ ]:
import json
from collections import OrderedDict

by_type = OrderedDict()
for r in records:
    by_type.setdefault(r['task_type'], r)
for i, (t, r) in enumerate(list(by_type.items())[:3], 1):
    print(f'\n===== Example {i} ({t}) =====')
    print(json.dumps(r, indent=2)[:1200])


## 4. The JSONL schema

Every record is one JSON object on one line. Keys:

```
id              stable id derived from task_type + instruction
task_type       architecture_qa | code_navigation | debugging | tool_use | refusal
instruction     the user-facing question
input           extra user context (usually empty for these templates)
output          the gold answer
evidence        list of {file, line_start, line_end, symbol, relation, confidence}
negative_examples  what the model must NOT say (empty for v0)
metadata        generated_by, difficulty, license, split, component, repo
```

The `evidence` array is the load-bearing field: every claim in `output` must be traceable to one of those entries.


## 5. Write the JSONL file


In [ ]:
from src.dataset.jsonl_writer import write_jsonl

out_path = DATASETS_OUT_DIR / 'mini_repo_sft_v0.jsonl'
n = write_jsonl(out_path, records)
print(f'wrote {n} records to {out_path}')
print('first line:')
print(out_path.read_text(encoding='utf-8').splitlines()[0][:300])


## Summary

You produced an SFT JSONL dataset purely from graph queries. Every example carries evidence pointing at a real entity, which is what makes the dataset honest. Notebook 07 shows the training recipe that would consume this file.


## Exercises

1. Add a new template (e.g. *header-includes*: which Q/K headers does a class include?) to `TEMPLATES` and verify new records appear.
2. Use `train_test_split` (sklearn) to split `records` into train/eval by `id`. Write the split to two JSONL files.
3. Build a histogram of `output` token counts using notebook 05's tokenizer. Reject any example over 200 tokens.
